# Snack profiles by price tier across 13 attributes

**Vertical Line Graph** · a worked example from the StatNex teaching guide

**The story.** A consumer panel scored 56 snacks from 1 to 10 on 13 attributes. Averaged by price tier, the scores give one profile line each for Premium, Standard and Budget snacks across the 13 attributes.

**The question.** On which attributes do premium snacks really differ from budget ones?

**To run it**, choose **Runtime → Run all**. The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## Before the analysis

In [ ]:
# Each script saves its figures to the folder figures/, as it would on your computer.
# This shows each new figure under the cell that saved it.
from pathlib import Path

import matplotlib
from IPython import get_ipython
from IPython.display import Image, display

matplotlib.use("Agg")
shown = set()


def show_new_figures(*_):
    for path in sorted(Path("figures").glob("*.png"), key=lambda p: p.stat().st_mtime_ns):
        if (path, path.stat().st_mtime_ns) not in shown:
            shown.add((path, path.stat().st_mtime_ns))
            display(Image(filename=str(path)))


events = get_ipython().events
for old in [f for f in events.callbacks["post_run_cell"] if f.__name__ == "show_new_figures"]:
    events.unregister("post_run_cell", old)
events.register("post_run_cell", show_new_figures)

## The table

This cell saves the table `11_chart/vertical_line_graph.csv` (13 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
Price tier,Premium,Standard,Budget
Sweetness,5.15,6.07,5.24
Saltiness,5.35,4.87,5.9
Crunchiness,5.5,5.07,5.62
Rich flavour,7.25,5.87,3.38
Healthy image,5.8,5.07,5.33
Portion size,3.1,5.47,7.62
Packaging appeal,8.45,5.27,2.67
Brand awareness,7.0,5.07,4.19
Novelty,7.25,4.87,4.1
Shareability,3.1,5.53,7.57
Kid appeal,3.75,6.2,6.48
Value for money,2.7,5.33,8.1
Repurchase intent,4.65,5.8,5.9

## Libraries

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.width", 120)

## The data

In [ ]:
# One row per item, one column per series. Text and blank cells are missing; every
# item needs at least one number.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
series = ["Premium", "Standard", "Budget"]
item_name = "Price tier"
items = data[item_name].astype(str)
values = data[series].apply(pd.to_numeric, errors="coerce")

## Data for the chart

In [ ]:
# Dummy 1 and Dummy 2 are the scaffolding Excel needs to draw this chart: each item's
# position counted from the top (0.5, 1.5, ...) and the baseline of the bars.
table = values.copy()
table.insert(0, item_name, items)
table["Dummy 1"] = np.arange(len(table)) + 0.5
table["Dummy 2"] = 0
print(table.to_string(index=False))

## Spread between the series

In [ ]:
# For each item, the lowest and highest series and the gap between them -- the width
# between the lines on the chart -- largest gap first. Blank cells are skipped. When
# the series are group averages, a gap shows where the averages differ, not whether
# the difference is larger than chance: that needs the spread within each group and
# a test.
spread = pd.DataFrame({"item": items, "minimum": values.min(axis=1),
                       "maximum": values.max(axis=1)})
spread["spread"] = spread["maximum"] - spread["minimum"]
spread["highest series"] = values.idxmax(axis=1)
spread = spread.iloc[np.argsort(spread["spread"].to_numpy())[::-1]]
print()
print(spread.to_string(index=False))

## Vertical line graph

In [ ]:
# Items run down the page in the order of the table, values across.
Path("figures").mkdir(exist_ok=True)
positions = np.arange(len(items))
fig, ax = plt.subplots(figsize=(7, 0.45 * len(items) + 1.5))
for name in series:
    ax.plot(values[name], positions, marker="o", label=name)
ax.set_yticks(positions, items)
ax.invert_yaxis()
ax.legend(fontsize=8)
fig.savefig("figures/vertical_line_graph.png", dpi=120, bbox_inches="tight")

## What the result says

Each line is one price tier, and each row is one attribute. Where the lines are far apart, the tiers differ. The spread is the gap between the highest and the lowest tier on one attribute.

The largest spread is 5.78, for packaging appeal: Premium scores 8.45 and Budget 2.67. Premium also leads on rich flavour and novelty, both at 7.25. Budget leads on value for money (8.1 against 2.7), portion size and shareability. The smallest spreads are for crunchiness (0.55) and healthy image (0.73), where the tiers barely differ.

But on repurchase intent Budget scores 5.9 and Premium only 4.65. Premium snacks look better, yet budget snacks are the ones people more often plan to buy again.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.